# PAM Rodeo noise presentation

Reads the results (`bin_corr.csv`, `mode_var_plots.csv`, `summary_meta.json`) and saved PNGs
produced by `glider_noise_stats_plots.ipynb`, and assembles the summary PowerPoint deck. Run
`glider_data_explore.ipynb` then `glider_noise_stats_plots.ipynb` first.

In [1]:
import pandas as pd
import os
import json

import config
import pptx_utils as pptxu  # aliased so it doesn't shadow the python-pptx package name

glider_id = config.GLIDER_ID
OUT_DIR = config.OUT_DIR          # where the final .pptx gets saved
DATA_DIR = config.DATA_DIR        # where notebook 2's CSV/JSON results live
FIGURES_DIR = config.FIGURES_DIR  # where notebook 2's PNGs live

with open(os.path.join(DATA_DIR, f'{glider_id}_summary_meta.json')) as f:
    summary_meta = json.load(f)

mode_order = summary_meta['mode_order']
BIN_VARS = summary_meta['bin_vars']

overall_corr_df = pd.read_csv(os.path.join(DATA_DIR, f'{glider_id}_overall_corr.csv'))
bin_corr_df = pd.read_csv(os.path.join(DATA_DIR, f'{glider_id}_bin_corr.csv'))
qc_impact_df = pd.read_csv(os.path.join(DATA_DIR, f'{glider_id}_qc_impact.csv'))
mode_var_plots_df = pd.read_csv(os.path.join(DATA_DIR, f'{glider_id}_mode_var_plots.csv'))
mode_var_plots = {
    (row['mode'], row['var']): {'spectrum': row['spectrum'], 'box': row['box']}
    for _, row in mode_var_plots_df.iterrows()
}

print(f'Loaded results for {glider_id}: {len(mode_order)} modes, {len(bin_corr_df)} bin-corr rows')

Loaded results for sg607: 9 modes, 63 bin-corr rows


In [2]:
def build_noise_deck(figures_dir, save_dir, glider_id, mode_order, mode_var_plots, bin_corr_df,
                      overall_corr_df, qc_impact_df, summary_meta, output_path=None):
    if output_path is None:
        output_path = f'{glider_id}_pam_noise_summary.pptx'

    prs = pptxu.new_presentation()

    def add_image_slide_safe(title, img_path, description):
        """Skip (with a warning) instead of crashing the whole deck if a figure never got
        generated - e.g. a notebook 2 plotting cell failed on this environment (no internet
        for cartopy's map tiles, etc.) so the PNG was never saved."""
        if not os.path.exists(img_path):
            print(f'Skipping "{title}" - figure not found: {img_path}')
            return
        pptxu.add_image_slide(prs, title, img_path, description)

    qc_status = 'QC applied' if summary_meta.get('qc_applied') else 'No QC applied'
    pptxu.add_title_slide(prs, f'{glider_id} PAM Noise Analysis',
                           f'{len(mode_order)} mission phases analyzed | {qc_status}')

    bullets = [
        f'{summary_meta["n_rows"]:,} science rows across {len(mode_order)} mission phases: {", ".join(mode_order)}',
        f'Overall energy-mean broadband SPL: {summary_meta["energy_mean_broadband"]:.1f} dB re: 1 μPa '
        f'(min {summary_meta["min_broadband"]:.1f}, max {summary_meta["max_broadband"]:.1f})',
        f'Variables binned: {", ".join(BIN_VARS)}',
        f'QC status: {qc_status}',
    ]
    if not qc_impact_df.empty:
        n_total = int(qc_impact_df['n_total'].sum())
        n_dropped = int(qc_impact_df['n_dropped'].sum())
        pct_dropped = 100 * n_dropped / n_total if n_total else 0
        bullets.append(
            f'depth_mask_flag: {n_dropped:,} of {n_total:,} rows flagged ({pct_dropped:.1f}%)'
            + (' - dropped from this analysis' if summary_meta.get('qc_applied') else ' - not dropped (QC off)')
        )
    if not overall_corr_df.empty:
        top = overall_corr_df.iloc[0]
        bullets.append(
            f'Strongest overall correlation with broadband SPL: {top["var"]} '
            f'(r={top["r"]:.2f}, n={int(top["n"]):,})'
        )
    if not bin_corr_df.empty:
        strongest = bin_corr_df.loc[bin_corr_df['r'].abs().idxmax()]
        bullets.append(
            f'Strongest per-mode correlation: {strongest["var"]} during "{strongest["mode"]}" '
            f'(r={strongest["r"]:.2f}, n={int(strongest["n"]):,})'
        )
    pptxu.add_bullet_slide(prs, 'Summary', bullets)

    pptxu.add_section_slide(prs, 'Full-mission overview')
    add_image_slide_safe('Depth over time by mission phase',
                          os.path.join(figures_dir, f'{glider_id}_time_depth_by_mode.png'),
                          'Full-deployment dive profile, colored by mission phase.')
    add_image_slide_safe('Glider track: phase & QC',
                          os.path.join(figures_dir, f'{glider_id}_track_by_mode.png'),
                          'Full-deployment glider track (longitude vs latitude): left colored by mission '
                          'phase, right highlighting rows flagged by the depth_mask_flag QC check.')
    add_image_slide_safe('Energy-mean spectrum by mission phase',
                          os.path.join(figures_dir, f'{glider_id}_spectrum_by_mode.png'),
                          'Energy-mean hybrid millidecade noise spectrum for each mission phase (levels '
                          'averaged in linear power, then converted back to dB) - frequency (log) vs SPL, '
                          'one line per mode.')
    add_image_slide_safe('Broadband SPL by mission phase',
                          os.path.join(figures_dir, f'{glider_id}_broadband_boxplot_by_mode.png'),
                          'Distribution of broadband SPL for each mission phase. Green triangles mark the '
                          'energy mean.')
    add_image_slide_safe('Broadband SPL before vs after QC',
                          os.path.join(figures_dir, f'{glider_id}_qc_before_after_boxplot.png'),
                          'One pair of boxes per mission phase: all data vs depth_mask_flag-filtered '
                          '("QC\'d"), green triangles = energy mean. If QC is off, each pair matches - '
                          'nothing was dropped.')
    add_image_slide_safe('Whole-deployment correlation matrix',
                          os.path.join(figures_dir, f'{glider_id}_overall_corr_matrix.png'),
                          'Pearson correlation between every CTD/science variable and broadband SPL, '
                          'all mission phases combined.')
    add_image_slide_safe('Broadband SPL correlation strength by variable and mode',
                          os.path.join(figures_dir, f'{glider_id}_broadband_correlation_by_mode.png'),
                          'Real per-point Pearson r between broadband SPL and each raw variable, '
                          'grouped by mission phase.')
    add_image_slide_safe('Correlation matrix by mission phase',
                          os.path.join(figures_dir, f'{glider_id}_corr_matrix_by_mode.png'),
                          'Full CTD/science variable + broadband SPL correlation matrix, computed '
                          'separately within each mission phase, for direct side-by-side comparison.')

    for mode in mode_order:
        pptxu.add_section_slide(prs, mode)
        for var in BIN_VARS:
            plots = mode_var_plots.get((mode, var))
            if not plots:
                continue
            add_image_slide_safe(f'{mode} - spectrum by {var} bin', plots['spectrum'],
                                  f'Energy-mean noise spectrum during "{mode}" (frequency, log scale, vs SPL), '
                                  f'one line per {var} bin.')
            add_image_slide_safe(f'{mode} - SPL by {var} bin', plots['box'],
                                  f'Broadband SPL distribution during "{mode}", split by {var} bin (green '
                                  f'triangles = energy mean). Title shows the real per-point Pearson r '
                                  f'between raw {var} and raw SPL for this mode.')

    prs.save(os.path.join(save_dir, output_path))
    print(f'Saved {os.path.join(save_dir, output_path)}')

build_noise_deck(FIGURES_DIR, OUT_DIR, glider_id, mode_order, mode_var_plots, bin_corr_df,
                  overall_corr_df, qc_impact_df, summary_meta)

Saved ./noise_analysis_outputs\sg607\sg607_pam_noise_summary.pptx
